# NeuroGeo-Field: Interactive Demonstration Dashboard
## Operational Address Inspection, Evidence Cards & Offline Field Guidance
**OpenIIT '26 — Problem Statement 3 (PS3)**

---

### Dashboard Purpose
This demonstration notebook provides a pure offline visual inspection interface for the 100 benchmark addresses evaluated by **NeuroGeo-Field**.

For any selected `address_id`, this dashboard displays:
1. **Raw Address Text & Parsed Entity Attributes** (Town, Locality, Street, House, Landmark)
2. **Commercial Baseline vs. NeuroGeo-Field Spatial Plot** in metric Cartesian coordinates $(x, y)$
3. **Calibrated Uncertainty Radius Circle** ($R_T$) based on the evidence status tier
4. **Field Guidance & Deterministic Navigation Instructions** (bearing, distance, compass heading)
5. **Auditable Evidence Card** (status tier, candidate scores, score margins, positive/negative evidence, caution notes)
6. **Surveyed Ground Truth** (**Evaluation-only ground truth**, displayed strictly for benchmark verification)

Four fixed representative case studies are featured:
- **Case 1: Visited Doorstep** (`AD002707` — `DOORSTEP_VERIFIED`, $C_2$ selected, 5.69m error)
- **Case 2: Cold-Start Street Transfer** (`AD002159` — `STREET_RESOLVED`, $C_4$ selected, 323.96m error)
- **Case 3: Locality-Bounded Centroid** (`AD000527` — `LOCALITY_BOUNDED`, $C_5$ selected, 141.89m error)
- **Case 4: Failure / Baseline Fallback** (`AD001909` — `BASELINE_FALLBACK`, $C_1$ selected, 4807.68m error)


In [ ]:
# Dashboard Setup & Imports
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

if sys.stdout and hasattr(sys.stdout, "reconfigure"):
    try:
        sys.stdout.reconfigure(encoding="utf-8")
    except Exception:
        pass

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from neurogeo.config import PATHS
from neurogeo.navigator import navigate, build_evidence_card

print("Interactive Dashboard components ready.")


In [ ]:
# Ingest Authoritative Artifacts
df_preds = pd.read_csv(PATHS.phase4_predictions_csv)
df_addr = pd.read_csv(PATHS.addresses_csv)
df_base = pd.read_csv(PATHS.baseline_geocodes_csv)
df_surveyed = pd.read_csv(PATHS.surveyed_addresses_csv)
df_entities = pd.read_parquet(PATHS.address_entities_parquet) if PATHS.address_entities_parquet.exists() else pd.DataFrame()
df_medoids = pd.read_parquet(PATHS.metrics_dir / "address_doorstep_medoids.parquet") if (PATHS.metrics_dir / "address_doorstep_medoids.parquet").exists() else pd.DataFrame()
df_candidates = pd.read_parquet(PATHS.metrics_dir / "candidate_scores.parquet") if (PATHS.metrics_dir / "candidate_scores.parquet").exists() else pd.DataFrame()

with open(PATHS.metrics_dir / "phase5_evidence_cards.json") as f:
    evidence_cards = {c["address_id"]: c for c in json.load(f)}

# Merge benchmark lookup
df_merged = df_preds.merge(df_addr[["address_id", "address_text", "town_id"]], on="address_id")
df_merged = df_merged.merge(df_base[["address_id", "geocoder_x", "geocoder_y", "precision"]], on="address_id")
df_merged["base_error_m"] = np.sqrt((df_merged["geocoder_x"] - df_merged["surveyed_x"])**2 + (df_merged["geocoder_y"] - df_merged["surveyed_y"])**2)

print(f"Loaded {len(df_merged)} benchmark addresses for interactive inspection.")


In [ ]:
def inspect_benchmark_address(address_id: str):
    """Render complete interactive evidence, spatial plot, and navigation card for an address."""
    match = df_merged[df_merged["address_id"] == address_id]
    if match.empty:
        print(f"Error: Address ID {address_id} not found in benchmark.")
        return
    row = match.iloc[0]
    card = evidence_cards.get(address_id, {})
    
    print("=" * 80)
    print(f"NEUROGEO-FIELD INSPECTION: {address_id} ({row['split'].upper()} SPLIT)")
    print("=" * 80)
    print(f"Raw Address:     {row['address_text']}")
    print(f"Town ID:         {row['town_id']}")
    print(f"Status Tier:     {row['status_tier']}")
    print(f"Candidate:       {row['selected_candidate_type']} (Total candidates: {row['candidate_count']})")
    print(f"Score Margin:    {row['score_margin']:.4f} (Score: {row['selected_score']:.4f} vs Runner-Up: {row['runner_up_score']:.4f})")
    print(f"Predicted Pin:   ({row['predicted_x']:.1f}, {row['predicted_y']:.1f})")
    print(f"Confidence Rad:  {row['confidence_radius_m']:.2f} m ({row['calibration_tier']})")
    print(f"Commercial Base: ({row['geocoder_x']:.1f}, {row['geocoder_y']:.1f}) | Baseline Error: {row['base_error_m']:.2f} m")
    print(f"NeuroGeo Error:  {row['error_vs_surveyed_m']:.2f} m (Improvement: {row['base_error_m'] - row['error_vs_surveyed_m']:+.2f} m)")
    print(f"Covered by Rad:  {row['is_covered']}")
    print("-" * 80)
    print(f"Positive Evid:   {card.get('positive_evidence', [])}")
    print(f"Negative Evid:   {card.get('negative_evidence', [])}")
    if card.get('caution'):
        print(f"Caution:         {card.get('caution')}")
    print(f"Explanation:     {card.get('explanation')}")
    print("=" * 80)
    
    # Generate Navigation Instruction from Town Origin (0, 0)
    pred_dict = {
        "address_id": address_id,
        "predicted_x": row["predicted_x"],
        "predicted_y": row["predicted_y"],
        "status_tier": row["status_tier"],
        "confidence_radius_m": row["confidence_radius_m"]
    }
    nav_inst = navigate(0.0, 0.0, pred_dict, negative_evidence=card.get("negative_evidence", []))
    mode_name = getattr(nav_inst.navigation_mode, "value", str(nav_inst.navigation_mode))
    print("\nOFFLINE FIELD NAVIGATION GUIDANCE (From Town Center Origin):")
    print(f"  Mode:        {mode_name}")
    print(f"  Distance:    {nav_inst.distance_m:.1f} m, Bearing: {nav_inst.bearing_deg:.1f}° ({nav_inst.direction})")
    print(f"  Instruction: {nav_inst.instruction}")
    if nav_inst.caution:
        print(f"  Caution:     {nav_inst.caution}")
    print("=" * 80)
    
    # Spatial Visualization
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    pred_x, pred_y = row["predicted_x"], row["predicted_y"]
    base_x, base_y = row["geocoder_x"], row["geocoder_y"]
    surv_x, surv_y = row["surveyed_x"], row["surveyed_y"]
    rad_m = row["confidence_radius_m"]
    
    # 1. Uncertainty Circle
    tier_colors = {
        "DOORSTEP_VERIFIED": "#27ae60",
        "STREET_RESOLVED": "#2980b9",
        "LOCALITY_BOUNDED": "#f39c12",
        "BASELINE_FALLBACK": "#e74c3c"
    }
    c_color = tier_colors.get(row["status_tier"], "#34495e")
    circle = plt.Circle((pred_x, pred_y), rad_m, color=c_color, fill=True, alpha=0.12,
                        edgecolor=c_color, linestyle="--", linewidth=1.5,
                        label=f"Calibrated P90 Radius ({rad_m:.1f}m)")
    ax.add_patch(circle)
    
    # 2. Commercial Baseline
    ax.scatter(base_x, base_y, color="#c0392b", s=130, marker="x", linewidths=3, zorder=5,
               label=f"Commercial Baseline (Err: {row['base_error_m']:.1f}m)")
    ax.annotate("Baseline", (base_x, base_y), xytext=(-15, -18), textcoords="offset points",
                fontsize=9, color="#962d22", fontweight="bold")
    
    # 3. NeuroGeo Prediction
    ax.scatter(pred_x, pred_y, color=c_color, s=150, marker="o", edgecolors="black", linewidths=1.5, zorder=6,
               label=f"NeuroGeo {row['selected_candidate_type']} (Err: {row['error_vs_surveyed_m']:.1f}m)")
    ax.annotate(f"NeuroGeo ({row['selected_candidate_type']})", (pred_x, pred_y), xytext=(10, 10),
                textcoords="offset points", fontsize=9, color=c_color, fontweight="bold")
    
    # 4. Surveyed Ground Truth (EVALUATION-ONLY)
    ax.scatter(surv_x, surv_y, color="#d35400", s=140, marker="*", edgecolors="black", linewidths=1, zorder=7,
               label="Evaluation-only ground truth")
    ax.annotate("Surveyed Truth", (surv_x, surv_y), xytext=(-25, 12),
                textcoords="offset points", fontsize=9, color="#ba4a00", fontweight="bold")
    
    # Connecting vectors
    ax.plot([pred_x, surv_x], [pred_y, surv_y], color=c_color, linestyle=":", alpha=0.8, linewidth=1.5)
    ax.plot([base_x, surv_x], [base_y, surv_y], color="#c0392b", linestyle=":", alpha=0.5, linewidth=1.2)
    
    max_span = max(rad_m, abs(pred_x - base_x), abs(pred_y - base_y), abs(pred_x - surv_x), abs(pred_y - surv_y)) + 100
    ax.set_xlim(pred_x - max_span, pred_x + max_span)
    ax.set_ylim(pred_y - max_span, pred_y + max_span)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Cartesian X (meters)")
    ax.set_ylabel("Cartesian Y (meters)")
    ax.set_title(f"Spatial Case Study: {address_id} [{row['status_tier']}]\nError: {row['error_vs_surveyed_m']:.1f}m vs Baseline: {row['base_error_m']:.1f}m",
                 fontweight="bold")
    ax.legend(frameon=True, loc="upper right", fontsize=8.5)
    plt.tight_layout()
    plt.show()

print("Interactive inspection function defined: `inspect_benchmark_address(address_id)`")


---
## Case Study 1: Visited Doorstep Address (`AD002707`)
- **Status Tier**: `DOORSTEP_VERIFIED`
- **Selected Candidate**: $C_2$ (Integrity-Weighted Doorstep Medoid)
- **Outcome**: The commercial baseline suffered a **1094.83m error** due to postal code centroid fallback. NeuroGeo-Field fused direct verified visits to pin the doorstep with **5.69m error** (a 1,089m improvement!).


In [ ]:
inspect_benchmark_address("AD002707")


---
## Case Study 2: Cold-Start Street Knowledge Transfer (`AD002159`)
- **Status Tier**: `STREET_RESOLVED`
- **Selected Candidate**: $C_4$ (Hierarchical Street Neighbor Weighted Centroid)
- **Outcome**: This address had **zero prior visits**. The parser extracted `Gali No. 12` in `Ambedkar Nagar`. The spatial graph transferred coordinates from 2 visited neighbor accounts (`AD001662` and `AD002112`), reducing error to **323.96m** (vs. 412.41m baseline) within a calibrated **615.93m** radius.


In [ ]:
inspect_benchmark_address("AD002159")


---
## Case Study 3: Locality-Bounded Centroid (`AD000527`)
- **Status Tier**: `LOCALITY_BOUNDED`
- **Selected Candidate**: $C_5$ (Locality Polygon Centroid)
- **Outcome**: An address with resolved locality (`Azad Mohalla`) but no verified street or doorstep check-ins. The system selects the administrative locality centroid with a calibrated **559.95m** uncertainty radius, achieving **141.89m error** (vs. 217.20m baseline).


In [ ]:
inspect_benchmark_address("AD000527")


---
## Case Study 4: Failure / Baseline Fallback (`AD001909`)
- **Status Tier**: `BASELINE_FALLBACK`
- **Selected Candidate**: $C_1$ (Commercial Baseline Fallback)
- **Outcome**: This address (`#448 6th cr 8th mn kaveripura - 960104`) had no field visits, no resolved locality polygon, and no street neighbors. The commercial geocoder snapped to a postal pincode centroid **4807.68m** away. NeuroGeo-Field falls back to baseline, surfaces an explicit `BASELINE_FALLBACK` flag, and flags low confidence in the navigation instruction.


In [ ]:
inspect_benchmark_address("AD001909")


---
## Interactive Address Browser

Use the snippet below to inspect any address from the 100 benchmark addresses. If `ipywidgets` is installed, an interactive dropdown will appear; otherwise, you can call `inspect_benchmark_address(address_id)` directly with any ID from `df_merged['address_id']`.


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    
    dropdown = widgets.Dropdown(
        options=sorted(df_merged["address_id"].tolist()),
        value="AD002707",
        description="Address ID:",
        disabled=False
    )
    
    def on_change(change):
        if change["type"] == "change" and change["name"] == "value":
            inspect_benchmark_address(change["new"])
            
    dropdown.observe(on_change)
    display(dropdown)
    inspect_benchmark_address(dropdown.value)
except ImportError:
    print("ipywidgets not installed in current environment. Running default demonstration on AD002854:")
    inspect_benchmark_address("AD002854")
